In [9]:
import pandas as pd
import requests
import re
import time

from bs4 import BeautifulSoup
from urllib.parse import quote_plus, urljoin
from pathlib import Path

In [10]:
main_file = Path(r"D:\Proyek FEB\pakai ini yang terbaru.xlsx")

df_main = pd.read_excel(main_file)

df_main.head()

,No,Title,Authors,Journal,SCOPUS Q,Tahun,Volume / Issue,LINK DOI/ARTIKEL,SCOPUS SITASI,raw_key,q_source_checkpoint2,journal_key_strong,q_empty
0,1,Ethnic identity and internal migration decisio...,ILMIAWAN AUWALIN,Journal of Ethnic and Migration Studies,Q1,2019-01-11 00:00:00,"Vol. 14, Issue 13",10.1080/1369183X.2018.1561252,24.0,journal of ethnic and migration studies,scimago_2025_database,journal of ethnic and migration studies,False
1,2,The role of service quality within Indonesian ...,BADRI MUNIR SUKOCO,Journal of Islamic Marketing,Q2,2020-01-14 00:00:00,"Vol. 11, Issue 1",10.1108/JIMA-03-2017-0033,60.0,journal of islamic marketing,scimago_2025_database,journal of islamic marketing,False
2,3,Developing Islamic crowdfunding website platfo...,MUHAMAD NAFIK HADI RYANDONO,Journal of Islamic Marketing,Q2,2020-08-21 00:00:00,"Vol. 11, Issue 5",10.1108/JIMA-02-2019-0022,48.0,journal of islamic marketing,scimago_2025_database,journal of islamic marketing,False
3,4,Developing Islamic crowdfunding website platfo...,ACHSANIA HENDRATMI,Journal of Islamic Marketing,Q2,2020-08-21 00:00:00,"Vol. 11, Issue 5",10.1108/JIMA-02-2019-0022,48.0,journal of islamic marketing,scimago_2025_database,journal of islamic marketing,False
4,5,Developing Islamic crowdfunding website platfo...,PUJI SUCIA SUKMANINGRUM,Journal of Islamic Marketing,Q2,2020-08-21 00:00:00,"Vol. 11, Issue 5",10.1108/JIMA-02-2019-0022,48.0,journal of islamic marketing,scimago_2025_database,journal of islamic marketing,False


In [11]:
COL_AUTHOR = "Authors"

In [12]:
authors_df = (
    df_main[[COL_AUTHOR]]
    .dropna()
    .drop_duplicates()
    .rename(columns={COL_AUTHOR: "author_name"})
    .reset_index(drop=True)
)

authors_df.head()

,author_name
0,ILMIAWAN AUWALIN
1,BADRI MUNIR SUKOCO
2,MUHAMAD NAFIK HADI RYANDONO
3,ACHSANIA HENDRATMI
4,PUJI SUCIA SUKMANINGRUM


In [13]:
print("Jumlah author unik:", len(authors_df))

Jumlah author unik: 152


In [14]:
session = requests.Session()

session.headers.update({
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9,id;q=0.8",
})

In [15]:
def clean_text(value):
    if value is None:
        return None
    
    text = str(value).strip()
    text = re.sub(r"\s+", " ", text)
    
    return text if text else None


def parse_year(text):
    if not text:
        return None
    
    match = re.search(r"(19|20)\d{2}", str(text))
    
    if match:
        return int(match.group(0))
    
    return None


def parse_citation(text):
    if not text:
        return None
    
    text = str(text)
    
    match = re.search(r"(\d+)\s*cited", text, flags=re.IGNORECASE)
    
    if match:
        return int(match.group(1))
    
    match = re.search(r"(\d+)", text)
    
    if match:
        return int(match.group(1))
    
    return None


def parse_quartile(text):
    if not text:
        return None
    
    text = str(text).strip()
    
    if re.search(r"\bno-?Q\b", text, flags=re.IGNORECASE):
        return "no-Q"
    
    match = re.search(r"\bQ\s*([1-4])\b", text, flags=re.IGNORECASE)
    
    if match:
        return f"Q{match.group(1)}"
    
    return None

In [16]:
def get_sinta_id_from_author_name(author_name, sleep_time=1.5):
    search_url = f"https://sinta.kemdiktisaintek.go.id/authors?q={quote_plus(str(author_name))}"
    
    try:
        response = session.get(search_url, timeout=30)
        time.sleep(sleep_time)
        
        if response.status_code != 200:
            return {
                "author_name": author_name,
                "sinta_id": None,
                "profile_url": None,
                "status": "request_failed",
                "http_status": response.status_code,
                "search_url": search_url
            }
        
        soup = BeautifulSoup(response.text, "html.parser")
        page_text = soup.get_text(" ", strip=True)
        
        # Cara utama: ambil link /authors/profile/{id}
        profile_link = soup.select_one("a[href*='/authors/profile/']")
        
        if profile_link and profile_link.get("href"):
            profile_url = urljoin(search_url, profile_link.get("href"))
            match = re.search(r"/authors/profile/(\d+)", profile_url)
            sinta_id = match.group(1) if match else None
            
            return {
                "author_name": author_name,
                "sinta_id": sinta_id,
                "profile_url": profile_url,
                "status": "found",
                "http_status": response.status_code,
                "search_url": search_url
            }
        
        # Fallback: cari teks SINTA ID
        match = re.search(r"SINTA\s*ID\s*:\s*(\d+)", page_text, flags=re.IGNORECASE)
        
        if match:
            sinta_id = match.group(1)
            profile_url = f"https://sinta.kemdiktisaintek.go.id/authors/profile/{sinta_id}"
            
            return {
                "author_name": author_name,
                "sinta_id": sinta_id,
                "profile_url": profile_url,
                "status": "found",
                "http_status": response.status_code,
                "search_url": search_url
            }
        
        return {
            "author_name": author_name,
            "sinta_id": None,
            "profile_url": None,
            "status": "not_found",
            "http_status": response.status_code,
            "search_url": search_url
        }
    
    except Exception as e:
        return {
            "author_name": author_name,
            "sinta_id": None,
            "profile_url": None,
            "status": "error",
            "error": str(e),
            "search_url": search_url
        }

In [17]:
get_sinta_id_from_author_name("novrys suhardianto")

{'author_name': 'novrys suhardianto',
 'sinta_id': '6701690',
 'profile_url': 'https://sinta.kemdiktisaintek.go.id/authors/profile/6701690',
 'status': 'found',
 'http_status': 200,
 'search_url': 'https://sinta.kemdiktisaintek.go.id/authors?q=novrys+suhardianto'}

In [18]:
def extract_journal_from_item(item):
    """
    Ambil journal dari ar-meta.
    Hindari teks Q, Author Order, Creator, Year, Cited.
    """
    meta_texts = [
        clean_text(meta.get_text(" ", strip=True))
        for meta in item.select(".ar-meta")
    ]
    meta_texts = [m for m in meta_texts if m]
    
    for meta in meta_texts:
        lower = meta.lower()
        
        if re.search(r"\bq[1-4]\b", lower) or "no-q" in lower:
            continue
        
        if "author order" in lower:
            continue
        
        if "creator" in lower:
            continue
        
        if "cited" in lower:
            continue
        
        if parse_year(meta):
            continue
        
        return meta
    
    # Fallback: cari link selain title/year/cited/quartile
    for a in item.find_all("a"):
        text = clean_text(a.get_text(" ", strip=True))
        cls = " ".join(a.get("class", []))
        
        if not text:
            continue
        
        if "ar-year" in cls or "ar-cited" in cls or "ar-quartile" in cls:
            continue
        
        if parse_quartile(text):
            continue
        
        if parse_year(text):
            continue
        
        if "scopus.com/record" in str(a.get("href", "")):
            continue
        
        return text
    
    return None

In [19]:
def scrape_author_profile_2026(profile_url, author_name=None, sinta_id=None, sleep_time=1.5):
    try:
        response = session.get(profile_url, timeout=30)
        time.sleep(sleep_time)
        
        if response.status_code != 200:
            return {
                "status": "request_failed",
                "http_status": response.status_code,
                "profile_url": profile_url,
                "items": []
            }
        
        soup = BeautifulSoup(response.text, "html.parser")
        
        items = []
        
        for item in soup.select("div.ar-list-item"):
            title_el = item.select_one(".ar-title a")
            
            title = clean_text(title_el.get_text(" ", strip=True)) if title_el else None
            scopus_link = urljoin(profile_url, title_el.get("href")) if title_el and title_el.get("href") else None
            
            year_el = item.select_one(".ar-year")
            year_text = year_el.get_text(" ", strip=True) if year_el else None
            year = parse_year(year_text)
            
            # Ambil hanya tahun 2026
            if year != 2026:
                continue
            
            cited_el = item.select_one(".ar-cited")
            citation_text = cited_el.get_text(" ", strip=True) if cited_el else None
            citation = parse_citation(citation_text)
            
            item_text = item.get_text(" ", strip=True)
            scopus_q = parse_quartile(item_text)
            
            journal = extract_journal_from_item(item)
            
            items.append({
                "author_name": author_name,
                "sinta_id": sinta_id,
                "title": title,
                "journal": journal,
                "scopus_q": scopus_q,
                "scopus_citation": citation,
                "year": year,
                "scopus_link": scopus_link,
                "profile_url": profile_url,
                "raw_text": item_text
            })
        
        return {
            "status": "ok",
            "http_status": response.status_code,
            "profile_url": profile_url,
            "items": items
        }
    
    except Exception as e:
        return {
            "status": "error",
            "error": str(e),
            "profile_url": profile_url,
            "items": []
        }

In [20]:
author_result = get_sinta_id_from_author_name("novrys suhardianto")

profile_result = scrape_author_profile_2026(
    profile_url=author_result["profile_url"],
    author_name=author_result["author_name"],
    sinta_id=author_result["sinta_id"]
)

pd.DataFrame(profile_result["items"])

,author_name,sinta_id,title,journal,scopus_q,scopus_citation,year,scopus_link,profile_url,raw_text
0,novrys suhardianto,6701690,Generalist CEOs and the Cost of Debt: Evidence...,Gadjah Mada International Journal of Business,Q2,0,2026,https://www.scopus.com/record/display.uri?eid=...,https://sinta.kemdiktisaintek.go.id/authors/pr...,Generalist CEOs and the Cost of Debt: Evidence...
1,novrys suhardianto,6701690,The Role of Sustainability Assurance in Enhanc...,Journal of Risk and Financial Management,Q1,0,2026,https://www.scopus.com/record/display.uri?eid=...,https://sinta.kemdiktisaintek.go.id/authors/pr...,The Role of Sustainability Assurance in Enhanc...


In [21]:
def scrape_2026_publications_by_author_name(author_name):
    author_result = get_sinta_id_from_author_name(author_name)
    
    if author_result["status"] != "found":
        return {
            "author_name": author_name,
            "sinta_id": None,
            "profile_url": None,
            "status": author_result["status"],
            "items": []
        }
    
    profile_result = scrape_author_profile_2026(
        profile_url=author_result["profile_url"],
        author_name=author_name,
        sinta_id=author_result["sinta_id"]
    )
    
    return {
        "author_name": author_name,
        "sinta_id": author_result["sinta_id"],
        "profile_url": author_result["profile_url"],
        "status": profile_result["status"],
        "items": profile_result["items"]
    }

In [23]:
all_items = []
author_logs = []

for i, row in authors_df.iterrows():
    author_name = row["author_name"]
    
    print(f"{i+1}/{len(authors_df)} - {author_name}")
    
    result = scrape_2026_publications_by_author_name(author_name)
    
    items = result.get("items", [])
    
    author_logs.append({
        "author_name": author_name,
        "sinta_id": result.get("sinta_id"),
        "profile_url": result.get("profile_url"),
        "status": result.get("status"),
        "items_2026": len(items)
    })
    
    # Jika author tidak upload paper 2026, skip otomatis karena items kosong
    all_items.extend(items)
    
    if (i + 1) % 10 == 0:
        pd.DataFrame(all_items).to_excel("sinta_2026_publications_checkpoint.xlsx", index=False)
        # pd.DataFrame(author_logs).to_excel("sinta_2026_author_log_checkpoint.xlsx", index=False)

sinta_2026_df = pd.DataFrame(all_items)
# author_log_df = pd.DataFrame(author_logs)

sinta_2026_df.to_excel("sinta_2026_publications_raw.xlsx", index=False)
# author_log_df.to_excel("sinta_2026_author_log.xlsx", index=False)

print("Jumlah publikasi 2026 ditemukan:", len(sinta_2026_df))
sinta_2026_df.head()

1/152 - ILMIAWAN AUWALIN
2/152 - BADRI MUNIR SUKOCO
3/152 - MUHAMAD NAFIK HADI RYANDONO
4/152 - ACHSANIA HENDRATMI
5/152 - PUJI SUCIA SUKMANINGRUM
6/152 - MASMIRA KURNIAWATI
7/152 - TANTI HANDRIANA
8/152 - PRAPTINI YULIANTI
9/152 - RADITYA SUKMANA
10/152 - DAMAI NASUTION
11/152 - SONY KUSUMASONDJAJA
12/152 - NOORLAILIE SOEWARNO
13/152 - INDRIANAWATI USMAN
14/152 - BAYU ARIE FIANTO
15/152 - RUDI PURWONO
16/152 - M. KHOERUL MUBIN
17/152 - LILIK SUGIHARTI
18/152 - MIGUEL ANGEL ESQUIVIAS PADILLA
19/152 - ROSSANTO DWI HANDOYO
20/152 - Angga Erlando
21/152 - DIAN AGUSTIA
22/152 - YANI PERMATASARI
23/152 - IMAN HARYMAWAN
24/152 - MOH. NASIH
25/152 - AMALIA RIZKI
26/152 - I MADE NARSA
27/152 - BAMBANG TJAHJADI
28/152 - AHMAD RIZKI SRIDADI
29/152 - GIGIH PRIHANTONO
30/152 - TRI SIWI AGUSTINA
31/152 - IRHAM ZAKI
32/152 - DEBBY RATNA DANIEL
33/152 - EKO FAJAR CAHYONO
34/152 - SUHERMAN ROSYIDI
35/152 - Raras Kirana Wandira
36/152 - R. MOH. QUDSI FAUZI
37/152 - DENIZAR ABDURRAHMAN MI`RAJ
38/152 - R

,author_name,sinta_id,title,journal,scopus_q,scopus_citation,year,scopus_link,profile_url,raw_text
0,BADRI MUNIR SUKOCO,256099,Women's leadership on human capital performanc...,Journal of Intellectual Capital,Q1,0,2026,https://www.scopus.com/record/display.uri?eid=...,https://sinta.kemdiktisaintek.go.id/authors/pr...,Women's leadership on human capital performanc...
1,BADRI MUNIR SUKOCO,256099,Determinant factors of government change capab...,Journal of Cleaner Production,Q1,0,2026,https://www.scopus.com/record/display.uri?eid=...,https://sinta.kemdiktisaintek.go.id/authors/pr...,Determinant factors of government change capab...
2,PUJI SUCIA SUKMANINGRUM,5985272,A New Framework of Islamic Entrepreneurship: F...,Sage Open,Q1,0,2026,https://www.scopus.com/record/display.uri?eid=...,https://sinta.kemdiktisaintek.go.id/authors/pr...,A New Framework of Islamic Entrepreneurship: F...
3,MASMIRA KURNIAWATI,6054268,Unveiling the intellectual landscape of artifi...,Discover Artificial Intelligence,Q1,1,2026,https://www.scopus.com/record/display.uri?eid=...,https://sinta.kemdiktisaintek.go.id/authors/pr...,Unveiling the intellectual landscape of artifi...
4,MASMIRA KURNIAWATI,6054268,Correction: Unveiling the intellectual landsca...,Discover Artificial Intelligence,Q1,0,2026,https://www.scopus.com/record/display.uri?eid=...,https://sinta.kemdiktisaintek.go.id/authors/pr...,Correction: Unveiling the intellectual landsca...


In [24]:
def build_2026_output_like_main(sinta_2026_df):
    output = pd.DataFrame()
    
    output["No"] = None
    output["Title"] = sinta_2026_df["title"]
    output["Authors"] = sinta_2026_df["author_name"]
    output["Journal"] = sinta_2026_df["journal"]
    output["SCOPUS Q"] = sinta_2026_df["scopus_q"]
    output["Tahun"] = sinta_2026_df["year"]
    output["Volume / Issue"] = None
    output["LINK DOI/ARTIKEL"] = sinta_2026_df["scopus_link"]
    output["SCOPUS SITASI"] = sinta_2026_df["scopus_citation"]
    
    # Kolom tambahan audit, nanti bisa dihapus kalau tidak dibutuhkan
    output["SINTA ID"] = sinta_2026_df["sinta_id"]
    output["SINTA PROFILE"] = sinta_2026_df["profile_url"]
    
    return output

In [25]:
sinta_2026_output = build_2026_output_like_main(sinta_2026_df)

sinta_2026_output.head()

,No,Title,Authors,Journal,SCOPUS Q,Tahun,Volume / Issue,LINK DOI/ARTIKEL,SCOPUS SITASI,SINTA ID,SINTA PROFILE
0,NaN,Women's leadership on human capital performanc...,BADRI MUNIR SUKOCO,Journal of Intellectual Capital,Q1,2026,None,https://www.scopus.com/record/display.uri?eid=...,0,256099,https://sinta.kemdiktisaintek.go.id/authors/pr...
1,NaN,Determinant factors of government change capab...,BADRI MUNIR SUKOCO,Journal of Cleaner Production,Q1,2026,None,https://www.scopus.com/record/display.uri?eid=...,0,256099,https://sinta.kemdiktisaintek.go.id/authors/pr...
2,NaN,A New Framework of Islamic Entrepreneurship: F...,PUJI SUCIA SUKMANINGRUM,Sage Open,Q1,2026,None,https://www.scopus.com/record/display.uri?eid=...,0,5985272,https://sinta.kemdiktisaintek.go.id/authors/pr...
3,NaN,Unveiling the intellectual landscape of artifi...,MASMIRA KURNIAWATI,Discover Artificial Intelligence,Q1,2026,None,https://www.scopus.com/record/display.uri?eid=...,1,6054268,https://sinta.kemdiktisaintek.go.id/authors/pr...
4,NaN,Correction: Unveiling the intellectual landsca...,MASMIRA KURNIAWATI,Discover Artificial Intelligence,Q1,2026,None,https://www.scopus.com/record/display.uri?eid=...,0,6054268,https://sinta.kemdiktisaintek.go.id/authors/pr...


In [26]:
main_columns = df_main.columns.tolist()

sinta_2026_for_merge = sinta_2026_output.copy()

# Pastikan semua kolom utama ada
for col in main_columns:
    if col not in sinta_2026_for_merge.columns:
        sinta_2026_for_merge[col] = None

# Urutkan sesuai data utama
sinta_2026_for_merge = sinta_2026_for_merge[main_columns]

In [27]:
df_combined = pd.concat(
    [df_main, sinta_2026_for_merge],
    ignore_index=True
)

if "No" in df_combined.columns:
    df_combined["No"] = range(1, len(df_combined) + 1)

In [28]:
df_combined.to_excel("data 2026_combined.xlsx", index=False)

print("Jumlah data utama:", len(df_main))
print("Jumlah publikasi 2026 ditambahkan:", len(sinta_2026_for_merge))
print("Total akhir:", len(df_combined))

Jumlah data utama: 2153
Jumlah publikasi 2026 ditambahkan: 126
Total akhir: 2279


In [29]:
def normalize_title(value):
    if pd.isna(value):
        return ""
    
    text = str(value).lower().strip()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^a-z0-9\s]", "", text)
    return text.strip()

df_combined["title_key"] = df_combined["Title"].apply(normalize_title)
df_combined["author_key"] = df_combined["Authors"].astype(str).str.lower().str.strip()

duplicates = df_combined[
    df_combined.duplicated(subset=["title_key", "author_key"], keep=False)
].copy()

duplicates[["Title", "Authors", "Journal", "Tahun"]].sort_values(["Title", "Authors"])

,Title,Authors,Journal,Tahun
1340,CEO busyness and investment efficiency: eviden...,IMAN HARYMAWAN,Journal of Financial Reporting and Accounting,1905-07-15 00:00:00
2185,CEO busyness and investment efficiency: eviden...,IMAN HARYMAWAN,Journal of Financial Reporting and Accounting,2026
1921,Examining the role of Islamic environmental va...,RIRIN TRI RATNASARI,Journal of Islamic Marketing,1905-07-17 00:00:00
2197,Examining the role of Islamic environmental va...,RIRIN TRI RATNASARI,Journal of Islamic Marketing,2026
2151,From Crisis to Recovery: Understanding the Imp...,LILIK SUGIHARTI,Journal of Population and Social Studies,2025-02-22 00:00:00
2171,From Crisis to Recovery: Understanding the Imp...,LILIK SUGIHARTI,Journal of Population and Social Studies,2026
2152,From Crisis to Recovery: Understanding the Imp...,MIGUEL ANGEL ESQUIVIAS PADILLA,Journal of Population and Social Studies,2025-02-22 00:00:00
2173,From Crisis to Recovery: Understanding the Imp...,MIGUEL ANGEL ESQUIVIAS PADILLA,Journal of Population and Social Studies,2026
2115,Gender exclusion in succession on family busin...,PRAPTINI YULIANTI,Competitiveness Review,2025-01-27 00:00:00
2159,Gender exclusion in succession on family busin...,PRAPTINI YULIANTI,Competitiveness Review,2026
